# SMS Spam Detection: A Comparative Analysis of Machine Learning and Deep Learning Models

In [3]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import nltk
import re

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Conv1D,
    GlobalMaxPooling1D,
    Dense
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Download NLTK stopwords for text preprocessing
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
df = pd.read_csv("spam.csv", encoding='latin-1')
print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'spam.csv'

In [ ]:
# Keep only the two relevant columns
df = df[['v1', 'v2']]

# Rename columns for clarity
df.columns = ['label', 'message']

print(df.shape)
display(df.head())

In [ ]:
sns.countplot(x='label', data=df, palette=['#1a1916', '#c8b98a'])
plt.title("Distribution of Spam vs Ham")
plt.xlabel("Class")
plt.ylabel("Number of Messages")
plt.show()

print(df['label'].value_counts())
print(df['label'].value_counts(normalize=True).mul(100).round(1))

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # 1. Convert all text to lowercase
    text = text.lower()

    # 2. Remove anything that is not a letter
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # 3. Split into words
    words = text.split()

    # 4. Remove stopwords
    words = [word for word in words if word not in stop_words]

    # 5. Reconstruct the sentence
    return " ".join(words)

In [ ]:
df['clean_message'] = df['message'].apply(clean_text)

# Comparison before / after cleaning
print("BEFORE:", df['message'][0])
print("AFTER:", df['clean_message'][0])

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['clean_message']).toarray()

y = df['label'].map({'ham': 0, 'spam': 1})

print(f"Shape of X: {X.shape}")
print(f"Example vector (first 5 values): {X[0][:5]}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

print("=== Logistic Regression ===")
print(classification_report(y_test, pred_lr, target_names=['ham', 'spam']))

In [ ]:
svm = SVC(kernel='linear', probability=True)
svm.fit(X_train, y_train)
pred_svm = svm.predict(X_test)

print("=== SVM ===")
print(classification_report(y_test, pred_svm, target_names=['ham', 'spam']))

In [ ]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['clean_message'])
sequences = tokenizer.texts_to_sequences(df['clean_message'])

# All messages will have the same length (100 tokens)

X_dl = pad_sequences(sequences, maxlen=100)
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split( X_dl, y, test_size=0.2, random_state=42 )

In [ ]:
model = Sequential([
    # Layer 1: Embedding — each word → 64-dimensional vector
    Embedding(input_dim=5000, output_dim=64, input_length=100),
    # Layer 2: Convolution — detects patterns over 5 consecutive words
    Conv1D(filters=64, kernel_size=5, activation='relu'),
    # Layer 3: Global Max Pooling — keeps the strongest pattern
    GlobalMaxPooling1D(),
    # Layer 4: Dense — classification layer
    Dense(32, activation='relu'),
    # Layer 5: Output — spam probability (between 0 and 1)
    Dense(1, activation='sigmoid')
    ])
model.summary()
model.compile( optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'] )

In [ ]:
history = model.fit( X_train_dl, y_train_dl, epochs=5, batch_size=32, validation_split=0.2, verbose=1 )
loss, accuracy = model.evaluate(X_test_dl, y_test_dl)
print(f"CNN Accuracy: {accuracy:.4f}")

In [ ]:
def predict_message(text):
  cleaned = clean_text(text)
  seq = tokenizer.texts_to_sequences([cleaned])
  padded = pad_sequences(seq, maxlen=100)
  proba = model.predict(padded, verbose=0)[0][0]
  label = "SPAM" if proba > 0.5 else "HAM"

  print(f"Original Message: {text}")
  print(f"Cleaned Message: {cleaned}")
  print(f"Spam Probability: {proba:.4f}")
  print(f"Prediction: {label}")
  print("-" * 50)



In [ ]:
# Live tests
predict_message("Congratulations! You won a free iPhone. Call now!")
predict_message("Hey, are you coming to the meeting tomorrow?")
predict_message("You have been selected to receive a £500 reward.")